# Synthetic Operational Dataset Generation

This notebook generates a synthetic laboratory operations dataset used throughout the project.

Because real operational datasets are typically proprietary, simulated workflow events were created to demonstrate operational monitoring, KPI reporting, and risk-based work prioritization.

The objective is to create realistic operational patterns rather than reproduce a specific facility.

In [12]:
# imports
import pandas as pd, numpy as np, random
from datetime import datetime, timedelta
from pathlib import Path

## Reproducibility
# Random seeds are fixed to ensure reproducible results and consistent downstream analyses.
np.random.seed(123); random.seed(123)

# Settings
instruments = {
    "HPLC_01": "HPLC", "HPLC_02": "HPLC",
    "UV_01": "UV_Spectrometer",
    "BAL_01": "Analytical_Balance",
    "PH_01":  "pH_Meter",
    "LUM_01": "Luminometer"
}
tests = ["Assay","Impurity","pH Check","Weigh","Endotoxin"]
operators = ["AD","BR","CF","DG","EL"]
products  = ["Reagent_A","Reagent_B","Media_X","Media_Y"]

start_date = datetime(2024,1,1)
days = 365

# --- base daily workload (Poisson lambda) by instrument type
base_lam = {
    "HPLC":4.0,
    "UV_Spectrometer":3.2,
    "Analytical_Balance":6.5,
    "pH_Meter":5.0, "Luminometer":2.2
}

## Simulation Design

The simulation includes seasonal demand, weekend workload reduction, instrument-specific workload patterns, downtime events, operator/product variation, and result status.

These assumptions are included to create operational variability similar to laboratory workflow data.

In [13]:
# helper functions
def season_factor(dt):
    if dt.month in [3,4,5]:   return 1.25 + np.random.normal(0, 0.05)
    if dt.month in [9,10,11]: return 1.15 + np.random.normal(0, 0.05)
    return 0.85 + np.random.normal(0, 0.05)

def weekend_factor(weekday): return 0.45 if weekday >= 5 else 1.0

base_down = {"HPLC":0.04,"UV_Spectrometer":0.02,"Analytical_Balance":0.01,
             "pH_Meter":0.015,"Luminometer":0.025}

def downtime_prob(inst_type, dt):
    p = base_down[inst_type]
    if 24 <= int(dt.strftime("%U")) <= 30:
        p *= np.random.uniform(1.5, 2.2)
    p *= np.random.uniform(0.8, 1.2)
    return float(np.clip(p, 0.005, 0.25))


## Automation Scenario Assumptions

A simulated automation implementation occurs on July 1, 2024.

The scenario assumes:

- reduced manual data entry
- increased throughput
- shorter average turnaround times

These assumptions are intentionally incorporated into the simulation and are used to demonstrate how operational impacts can be measured and reported.

In [14]:
def period(dt):  # before vs after automation
    return "Before_Automation" if dt < datetime(2024,7,1) else "After_Automation"

def manual_prob(dt):
    if dt < datetime(2024,7,1):
        return float(np.clip(np.random.normal(0.35, 0.07), 0.18, 0.55))
    else:
        return float(np.clip(np.random.normal(0.12, 0.05), 0.02, 0.30))

def automation_lift(dt):
    if dt < datetime(2024,7,1): 
        return 1.0
    return float(np.clip(np.random.normal(1.32, 0.06), 1.18, 1.45))


## Generate Synthetic Operational Events

Each row represents one instrument run. The generated fields include:

sample ID | instrument | test type | duration | turnaround time | result | operator | product family |manual-entry status | automation phase | 
and downtime information.

In [15]:
rows = []
sample_counter = 1  # ensure unique sample IDs across the year

for i in range(days):
    dt = start_date + timedelta(days=i)
    wkday = dt.weekday()

    for inst_id, inst_type in instruments.items():
        lam = base_lam[inst_type] * season_factor(dt) * weekend_factor(wkday) * automation_lift(dt)
        lam *= np.random.uniform(0.9, 1.1)  # per-instrument drift
        n_runs = np.random.poisson(max(lam, 0.1))

        down_today = np.random.rand() < downtime_prob(inst_type, dt)
        down_reason = random.choice(["maintenance","calibration","software_error"]) if down_today else ""
        p_manual = manual_prob(dt)

        for _ in range(n_runs):
            dur = max(6, np.random.normal(45, 14))  # minutes
            result = np.random.choice(["pass","fail"], p=[0.95, 0.05])

            # NEW: unique sample and TAT
            sample_id = f"S{dt.year}-{sample_counter:05d}"
            sample_counter += 1
            mean_tat = 30 if dt < datetime(2024, 7, 1) else 22  # hours, with variation
            turnaround_hours = max(6, np.random.normal(mean_tat, 5))

            rows.append({
                "sample_id": sample_id,
                "instrument_id": inst_id,
                "instrument_type": inst_type,
                "date": dt.date().isoformat(),
                "test_type": random.choice(tests),
                "duration_min": round(float(dur), 1),
                "turnaround_hours": round(float(turnaround_hours), 1),
                "result": result,
                "operator": random.choice(operators),
                "product_family": random.choice(products),
                "manual_entry": int(np.random.rand() < p_manual),
                "automation_phase": period(dt),
                "downtime_flag": int(down_today),
                "downtime_reason": down_reason
            })

df = pd.DataFrame(rows)
df.head()
            

,sample_id,instrument_id,instrument_type,date,test_type,duration_min,turnaround_hours,result,operator,product_family,manual_entry,automation_phase,downtime_flag,downtime_reason
0,S2024-00001,HPLC_01,HPLC,2024-01-01,Assay,35.5,29.5,pass,CF,Reagent_A,0,Before_Automation,0,
1,S2024-00002,HPLC_01,HPLC,2024-01-01,Weigh,38.8,27.8,pass,CF,Reagent_A,0,Before_Automation,0,
2,S2024-00003,HPLC_01,HPLC,2024-01-01,Assay,59.1,31.9,pass,DG,Media_X,0,Before_Automation,0,
3,S2024-00004,HPLC_01,HPLC,2024-01-01,pH Check,31.9,35.9,pass,AD,Reagent_B,1,Before_Automation,0,
4,S2024-00005,HPLC_01,HPLC,2024-01-01,Impurity,57.7,22.9,pass,CF,Media_X,0,Before_Automation,0,


## Export Output

This notebook produces:

- qc_instrument_usage.csv (raw operational events)

This file is consumed by downstream analysis and modeling notebooks.

In [16]:
from pathlib import Path

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "qc_instrument_usage.csv"

df.to_csv(output_file, index=False)

print(f"Saved {len(df):,} rows to {output_file}")

Saved 9,107 rows to data\qc_instrument_usage.csv
